# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [2]:
# Import necessary libraries
import os
import json
import numpy as np
import pandas as pd
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models
import keras.backend as K
from datetime import datetime


print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# Reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix, f1_score



TF: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Model Choice

As a baseline model, we implement the reference architecture provided in the challenge description:
a **3-layer 1D Convolutional Neural Network (CNN)**.

The architecture consists of:
- Two convolution + max-pooling blocks (each pooling with factor 10),
- One final convolution layer producing a 1 Hz probability sequence (90 values per window).

This baseline is appropriate because:
1. It is the official benchmark model of the challenge.
2. It processes raw PSG signals sampled at 100 Hz without handcrafted features.
3. It outputs a 1 Hz segmentation mask aligned with the provided ground-truth labels.
4. It provides a simple reference point for comparison with more complex architectures.

The model does not incorporate recurrence or attention mechanisms by design.




## Feature Selection

Each training example consists of:
- **8 PSG channels** sampled at **100 Hz**, resulting in input tensors of shape `(9000, 8)` for each 90-second window.
- A **binary segmentation mask** of shape `(90,)`, sampled at 1 Hz.

No handcrafted features are used.
The only preprocessing step applied is **signal normalization**, performed prior to training.

Subject identifiers are provided separately and are used exclusively to perform a
**subject-wise train/validation split**, preventing data leakage.



In [8]:
# Paths
IN_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\02_processed\nights_train_norm.h5"

with h5py.File(IN_PATH, "r") as f:
    Xn_train_nights = f["X_nights"][:]      # (22, 1800000, 8)
    y_nights        = f["y_nights"][:]      # (22, 18000)
    subj_order      = f["subject_ids"][:]   # (22,)

print(Xn_train_nights.shape, y_nights.shape, subj_order.shape)

(22, 1800000, 8) (22, 18000) (22,)


In [9]:
def chunk_generator(X_nights, y_nights, chunk_sec=300, stride_sec=60, fs=100):
    chunk_len = chunk_sec * fs
    stride_len = stride_sec * fs

    n_nights, T, C = X_nights.shape
    assert y_nights.shape[0] == n_nights
    assert y_nights.shape[1] == T // fs, f"y needs to have {T//fs} steps (1Hz) per night"

    for n in range(n_nights):
        X = X_nights[n]   # (T, 8) at 100Hz
        y = y_nights[n]   # (T//100,) at 1Hz
        for start in range(0, T - chunk_len + 1, stride_len):
            end = start + chunk_len

            X_chunk = X[start:end]
            y_chunk = y[start//fs : end//fs]

            # sanity: garante alinhamento perfeito
            if y_chunk.shape[0] != chunk_sec:
                continue

            yield X_chunk.astype("float32"), y_chunk.astype("float32")


In [10]:
#overlap = chunk − stride = 300 − 60 = 240 seconds 80% overlap
# stride = 120s → overlap = 180s (60%)
# stride = 150s → overlap = 150s (50%)
# stride = 300s → overlap = 0 (no overlap, but worse coverage)
CHUNK_SEC = 300  #5-minute (300s) windows 
STRIDE_SEC = 60
FS = 100

gen = chunk_generator(Xn_train_nights, y_nights, chunk_sec=CHUNK_SEC, stride_sec=STRIDE_SEC, fs=FS)
Xc, yc = next(gen)
print(Xc.shape, yc.shape)          # (30000, 8) (300,)
print("chunk % ones:", yc.mean()*100)


(30000, 8) (300,)
chunk % ones: 5.33333346247673


In [11]:
def make_tf_dataset(X_nights, y_nights, chunk_sec=300, stride_sec=60, fs=100,
                    batch_size=4, shuffle_buffer=1024):
    chunk_len = chunk_sec * fs

    output_signature = (
        tf.TensorSpec(shape=(chunk_len, 8), dtype=tf.float32),
        tf.TensorSpec(shape=(chunk_sec,), dtype=tf.float32),
    )

    ds = tf.data.Dataset.from_generator(
        lambda: chunk_generator(X_nights, y_nights, chunk_sec, stride_sec, fs),
        output_signature=output_signature
    )

    ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [12]:
SEED = 42
rng = np.random.default_rng(SEED)
idx = np.arange(22)
rng.shuffle(idx)

n_train = int(0.7 * len(idx))  # 15
train_idx = idx[:n_train]
val_idx   = idx[n_train:]

X_trainN = Xn_train_nights[train_idx]
y_trainN = y_nights[train_idx]
X_valN   = Xn_train_nights[val_idx]
y_valN   = y_nights[val_idx]

In [13]:
train_ds = make_tf_dataset(X_trainN, y_trainN, chunk_sec=300, stride_sec=60, batch_size=4)
val_ds   = make_tf_dataset(X_valN,   y_valN,   chunk_sec=300, stride_sec=60, batch_size=4, shuffle_buffer=256)

In [14]:
CHUNK_SEC = 300
STRIDE_SEC = 60
BATCH_SIZE = 4

train_ds = make_tf_dataset(Xn_train_nights, y_nights, chunk_sec=CHUNK_SEC, stride_sec=STRIDE_SEC, batch_size=BATCH_SIZE)

for xb, yb in train_ds.take(1):
    print("batch X:", xb.shape, "batch y:", yb.shape)  # (4, 30000, 8) e (4, 300)


batch X: (4, 30000, 8) batch y: (4, 300)


In [15]:
# 1) conferência de comprimento (sempre verdadeiro)
assert xb.shape[1] == 300 * 100
assert yb.shape[1] == 300

# 2) sanity do balanceamento no batch
print("true % ones no batch:", float(tf.reduce_mean(yb)) * 100)


true % ones no batch: 8.58333334326744


## Implementation

Below we implement the 3-layer CNN baseline as specified by the challenge.
Two Conv1D + MaxPool1D blocks downsample the 100 Hz signal to 1 Hz, and the final
Conv1D layer outputs class probabilities for each of the 90 seconds.


CNN Baseline model

In [ ]:
FS = 100
CHUNK_SEC = 300
N_SAMPLES = FS * CHUNK_SEC   # 30000
N_CHANNELS = 8

def build_baseline_cnn():
    model = models.Sequential([
        layers.Input(shape=(N_SAMPLES, N_CHANNELS)),

        layers.Conv1D(32, kernel_size=25, padding='same', activation='relu'),
        layers.MaxPooling1D(pool_size=10),  # 9000 → 900

        layers.Conv1D(64, kernel_size=25, padding='same', activation='relu'),
        layers.MaxPooling1D(pool_size=10),  # 900 → 90

        layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid'),
        layers.Lambda(lambda t: tf.squeeze(t, axis=-1))  # (batch, 90)
    ])
    return model

model = build_baseline_cnn()
model.summary()


Loss Function (Weighted BCE)

In [ ]:
positive_ratio = float(np.mean(y_trainN))
neg_ratio = 1.0 - positive_ratio
pos_weight = neg_ratio / (positive_ratio + 1e-12)
print("Positive ratio:", positive_ratio, "| pos_weight:", pos_weight)

def weighted_bce(pos_weight):
    def loss(y_true, y_pred):
        bce = K.binary_crossentropy(y_true, y_pred)
        w = 1.0 + (pos_weight - 1.0) * y_true
        return K.mean(bce * w)
    return loss


Compile & Train

In [ ]:
lr = 1e-4  
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss=weighted_bce(pos_weight),
    metrics=[tf.keras.metrics.AUC(curve="PR", name="auc_pr")]
)

OUT_DIR = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel"
os.makedirs(OUT_DIR, exist_ok=True)

ckpt_path = os.path.join(OUT_DIR, "baseline_best.keras")


run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = r"...\2_BaselineModel"
os.makedirs(OUT_DIR, exist_ok=True)

ckpt_best = os.path.join(OUT_DIR, f"{run_id}_best.keras")
ckpt_last = os.path.join(OUT_DIR, f"{run_id}_last.keras")



callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=ckpt_path, save_best_only=True, monitor="val_loss", verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.legend()
plt.show()


## Evaluation

Model performance is evaluated using an **event-based F1-score**, consistent with the
official metric of the Dreem Sleep Apnea Challenge.

Predictions are first reconstructed at **1 Hz over the full night** by aggregating
overlapping window-level outputs using an overlap-and-average strategy. The resulting
probability sequence is then thresholded to obtain a binary apnea mask.

Apnea events are extracted as contiguous segments in the binary mask and compared to the
ground-truth annotations using an **Intersection over Union (IoU)** criterion. A predicted
event is considered a true positive if its IoU with a reference event is greater than or
equal to **0.3**.

The final score is computed as an **event-level F1-score**, aggregating true positives,
false positives, and false negatives across all validation nights.

For completeness, window-level metrics such as accuracy and recall may be reported for
debugging purposes, but **they are not used for model selection**, as they do not reflect
the temporal structure of apnea events.



In [5]:
best_frozen = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\3lcnn_night\20251214_014656_BEST_frozen.keras"

model = tf.keras.models.load_model(best_frozen, compile = False)
print("Modelo carregado:", best_frozen)


Modelo carregado: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\3lcnn_night\20251214_014656_BEST_frozen.keras


In [6]:
def predict_nights_overlap_mean(
    model,
    X_nights,                 # (n_nights, 1800000, 8)
    chunk_sec=300,
    stride_sec=60,
    fs=100,
    batch_size=4
):
    n_nights, T, C = X_nights.shape
    chunk_len = chunk_sec * fs        # 30000
    stride_len = stride_sec * fs      # 6000
    T1 = T // fs                      # 18000

    y_sum = np.zeros((n_nights, T1), dtype=np.float32)
    y_cnt = np.zeros((n_nights, T1), dtype=np.float32)

    for n in range(n_nights):
        X = X_nights[n]  # (1800000, 8)
        starts = range(0, T - chunk_len + 1, stride_len)

        batch_X = []
        batch_ranges = []

        for start in starts:
            end = start + chunk_len
            x_chunk = X[start:end]          # (30000, 8)

            s1 = start // fs
            e1 = end // fs                  # 300 pontos

            batch_X.append(x_chunk)
            batch_ranges.append((s1, e1))

            if len(batch_X) == batch_size:
                preds = model.predict(np.stack(batch_X).astype("float32"), verbose=0)  # (B, 300)
                for p, (s1_, e1_) in zip(preds, batch_ranges):
                    y_sum[n, s1_:e1_] += p
                    y_cnt[n, s1_:e1_] += 1.0
                batch_X, batch_ranges = [], []

        if batch_X:
            preds = model.predict(np.stack(batch_X).astype("float32"), verbose=0)
            for p, (s1_, e1_) in zip(preds, batch_ranges):
                y_sum[n, s1_:e1_] += p
                y_cnt[n, s1_:e1_] += 1.0

    return y_sum / np.maximum(y_cnt, 1.0)


In [16]:
y_pred_val_nights = predict_nights_overlap_mean(
    model,
    X_valN,
    chunk_sec=300,
    stride_sec=60,
    fs=100,
    batch_size=4
)

print(y_pred_val_nights.shape)  # esperado: (n_val, 18000)


(7, 18000)


In [29]:

print("y_pred_val_nights:", y_pred_val_nights.shape)  # (n_val, 18000)
print("min/max/mean:", y_pred_val_nights.min(), y_pred_val_nights.max(), y_pred_val_nights.mean())


y_pred_val_nights: (7, 18000)
min/max/mean: 0.20183344 0.9724482 0.49136212


In [30]:
def extract_events_from_binary_mask(binary_mask, fs=1):
    binary_mask = np.asarray(binary_mask).astype(int)
    padded = np.concatenate([[0], binary_mask, [0]])
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0] / fs
    ends   = np.where(diff == -1)[0] / fs
    return [(float(s), float(e)) for s, e in zip(starts, ends)]

def jaccard_overlap(pred_events, true_events):
    # rows=true, cols=pred
    A = len(pred_events)
    B = len(true_events)
    if A == 0 or B == 0:
        return np.zeros((B, A), dtype=np.float32)

    p_start = np.array([s for s, e in pred_events])[None, :]
    p_end   = np.array([e for s, e in pred_events])[None, :]
    t_start = np.array([s for s, e in true_events])[:, None]
    t_end   = np.array([e for s, e in true_events])[:, None]

    inter = np.maximum(np.minimum(p_end, t_end) - np.maximum(p_start, t_start), 0.0)
    union = (p_end - p_start) + (t_end - t_start) - inter + 1e-12
    return (inter / union).astype(np.float32)

def tp_fp_fn(pred_events, true_events, min_iou=0.3):
    if len(pred_events) == 0:
        return 0, 0, len(true_events)
    if len(true_events) == 0:
        return 0, len(pred_events), 0

    iou = jaccard_overlap(pred_events, true_events)  # (n_true, n_pred)
    matched_true = int(np.any(iou >= min_iou, axis=1).sum())
    matched_pred = int(np.any(iou >= min_iou, axis=0).sum())

    tp = int(min(matched_true, matched_pred))
    fp = len(pred_events) - tp
    fn = len(true_events) - tp
    return tp, fp, fn

def event_f1_nights(y_pred_bin_nights, y_true_nights, min_iou=0.3):
    total_tp = total_fp = total_fn = 0
    for yp, yt in zip(y_pred_bin_nights, y_true_nights):
        pred_events = extract_events_from_binary_mask(yp, fs=1)
        true_events = extract_events_from_binary_mask(yt, fs=1)
        tp, fp, fn = tp_fp_fn(pred_events, true_events, min_iou=min_iou)
        total_tp += tp; total_fp += fp; total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-12)
    recall    = total_tp / (total_tp + total_fn + 1e-12)
    return float(2 * precision * recall / (precision + recall + 1e-12))


In [31]:
# y_valN deve ser (7, 18000) 0/1
y_true_val_nights = y_valN.astype(int)

best_t, best_f1 = None, -1
for t in np.linspace(0.20, 0.80, 31):
    yb = (y_pred_val_nights >= t).astype(int)
    f1 = event_f1_nights(yb, y_true_val_nights, min_iou=0.3)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("BEST threshold:", best_t)
print("BEST Event-F1:", best_f1)


BEST threshold: 0.5800000000000001
BEST Event-F1: 0.186413902053229


In [32]:
t = 0.58
yb = (y_pred_val_nights >= t).astype(int)

def count_events(mask_1hz):
    return len(extract_events_from_binary_mask(mask_1hz))

for i in range(yb.shape[0]):
    n_true = count_events(y_valN[i])
    n_pred = count_events(yb[i])
    print(f"night {i}: true_events={n_true:3d} | pred_events={n_pred:3d} | true%={y_valN[i].mean()*100:5.2f}% | pred%={yb[i].mean()*100:5.2f}%")


night 0: true_events= 20 | pred_events= 22 | true%= 1.39% | pred%= 0.37%
night 1: true_events= 96 | pred_events=  8 | true%= 9.56% | pred%= 0.09%
night 2: true_events=192 | pred_events=650 | true%=27.24% | pred%=51.87%
night 3: true_events= 21 | pred_events= 33 | true%= 1.63% | pred%= 0.57%
night 4: true_events= 16 | pred_events=  9 | true%= 2.90% | pred%= 0.18%
night 5: true_events= 18 | pred_events=  1 | true%= 1.63% | pred%= 0.04%
night 6: true_events=154 | pred_events= 26 | true%=13.77% | pred%= 0.43%


In [33]:
t = 0.57
yb = (y_pred_val_nights >= t).astype(int)

def count_events(mask_1hz):
    return len(extract_events_from_binary_mask(mask_1hz))

for i in range(yb.shape[0]):
    n_true = count_events(y_valN[i])
    n_pred = count_events(yb[i])
    print(f"night {i}: true_events={n_true:3d} | pred_events={n_pred:3d} | true%={y_valN[i].mean()*100:5.2f}% | pred%={yb[i].mean()*100:5.2f}%")

night 0: true_events= 20 | pred_events= 36 | true%= 1.39% | pred%= 0.58%
night 1: true_events= 96 | pred_events= 12 | true%= 9.56% | pred%= 0.14%
night 2: true_events=192 | pred_events=649 | true%=27.24% | pred%=54.78%
night 3: true_events= 21 | pred_events= 37 | true%= 1.63% | pred%= 0.65%
night 4: true_events= 16 | pred_events=  9 | true%= 2.90% | pred%= 0.19%
night 5: true_events= 18 | pred_events=  2 | true%= 1.63% | pred%= 0.07%
night 6: true_events=154 | pred_events= 29 | true%=13.77% | pred%= 0.57%


In [34]:
t = 0.56
yb = (y_pred_val_nights >= t).astype(int)

def count_events(mask_1hz):
    return len(extract_events_from_binary_mask(mask_1hz))

for i in range(yb.shape[0]):
    n_true = count_events(y_valN[i])
    n_pred = count_events(yb[i])
    print(f"night {i}: true_events={n_true:3d} | pred_events={n_pred:3d} | true%={y_valN[i].mean()*100:5.2f}% | pred%={yb[i].mean()*100:5.2f}%")

night 0: true_events= 20 | pred_events= 45 | true%= 1.39% | pred%= 0.81%
night 1: true_events= 96 | pred_events= 20 | true%= 9.56% | pred%= 0.31%
night 2: true_events=192 | pred_events=674 | true%=27.24% | pred%=58.25%
night 3: true_events= 21 | pred_events= 38 | true%= 1.63% | pred%= 0.77%
night 4: true_events= 16 | pred_events= 10 | true%= 2.90% | pred%= 0.19%
night 5: true_events= 18 | pred_events=  2 | true%= 1.63% | pred%= 0.11%
night 6: true_events=154 | pred_events= 45 | true%=13.77% | pred%= 0.80%


In [35]:
t = 0.55
yb = (y_pred_val_nights >= t).astype(int)

def count_events(mask_1hz):
    return len(extract_events_from_binary_mask(mask_1hz))

for i in range(yb.shape[0]):
    n_true = count_events(y_valN[i])
    n_pred = count_events(yb[i])
    print(f"night {i}: true_events={n_true:3d} | pred_events={n_pred:3d} | true%={y_valN[i].mean()*100:5.2f}% | pred%={yb[i].mean()*100:5.2f}%")

night 0: true_events= 20 | pred_events= 62 | true%= 1.39% | pred%= 1.18%
night 1: true_events= 96 | pred_events= 39 | true%= 9.56% | pred%= 0.61%
night 2: true_events=192 | pred_events=645 | true%=27.24% | pred%=81.56%
night 3: true_events= 21 | pred_events= 48 | true%= 1.63% | pred%= 1.02%
night 4: true_events= 16 | pred_events= 11 | true%= 2.90% | pred%= 0.24%
night 5: true_events= 18 | pred_events=  3 | true%= 1.63% | pred%= 0.12%
night 6: true_events=154 | pred_events= 83 | true%=13.77% | pred%= 1.34%


In [36]:
def postprocess_mask(mask, min_len=10, gap_fill=5):
    """
    mask: (T,) 0/1 em 1Hz
    min_len: remove eventos com duração < min_len segundos
    gap_fill: preenche buracos de zeros com duração <= gap_fill dentro de um evento
    """
    m = mask.astype(int).copy()
    T = len(m)

    # 1) fill small gaps (0-runs curtos entre 1s)
    i = 0
    while i < T:
        if m[i] == 0:
            j = i
            while j < T and m[j] == 0:
                j += 1
            gap = j - i
            left_one = (i - 1 >= 0 and m[i - 1] == 1)
            right_one = (j < T and m[j] == 1)
            if left_one and right_one and gap <= gap_fill:
                m[i:j] = 1
            i = j
        else:
            i += 1

    # 2) remove short events (1-runs curtos)
    i = 0
    while i < T:
        if m[i] == 1:
            j = i
            while j < T and m[j] == 1:
                j += 1
            run = j - i
            if run < min_len:
                m[i:j] = 0
            i = j
        else:
            i += 1

    return m


In [75]:
best = (-1, None)

for t in np.linspace(0.45, 0.75, 16):
    raw = (y_pred_val_nights >= t).astype(int)

    for min_len in [ 11, 12, 13, 14,16]:
        for gap_fill in [1, 2,3,5,7,8,9,10]:
            ypp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                            for i in range(raw.shape[0])])
            f1 = event_f1_nights(ypp, y_valN.astype(int), min_iou=0.3)

            if f1 > best[0]:
                best = (f1, (t, min_len, gap_fill))

print("BEST postproc Event-F1:", best[0])
print("params (t, min_len, gap_fill):", best[1])
#print porcentage of 1 
t = best[1][0]
min_len = best[1][1]
gap_fill = best[1][2]

raw = (y_pred_val_nights >= t).astype(int)
y_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                 for i in range(raw.shape[0])])
print("Percentage of 1s after post-processing:", y_pp.mean() * 100)

BEST postproc Event-F1: 0.24702058504826074
params (t, min_len, gap_fill): (0.5700000000000001, 12, 3)
Percentage of 1s after post-processing: 7.7746031746031745


In [70]:
#qual a duracao minima e max de enventos em x_train
min_event_durations = []
for i in range(y_nights.shape[0]):
    events = extract_events_from_binary_mask(y_nights[i], fs=1)
    durations = [e - s for s, e in events]
    if durations:
        min_event_durations.append(min(durations)) 

max_event_durations = []
for i in range(y_nights.shape[0]):
    events = extract_events_from_binary_mask(y_nights[i], fs=1)
    durations = [e - s for s, e in events]
    if durations:
        max_event_durations.append(max(durations))  
print("Maximum event durations in training nights:", max_event_durations)    
print("Minimum event durations in training nights:", min_event_durations)
def apply_postproc_from_probs(y_pred_nights, t, min_len, gap_fill):
    raw = (y_pred_nights >= t).astype(int)
    y_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                     for i in range(raw.shape[0])])
    return y_pp


Maximum event durations in training nights: [26.0, 63.0, 47.0, 55.0, 78.0, 38.0, 47.0, 44.0, 36.0, 53.0, 33.0, 39.0, 46.0, 29.0, 40.0, 20.0, 47.0, 25.0, 50.0, 34.0, 22.0, 35.0]
Minimum event durations in training nights: [9.0, 7.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 3.0, 1.0, 4.0, 1.0, 11.0, 1.0, 2.0, 2.0, 2.0, 1.0, 2.0, 2.0, 1.0]


In [ ]:
t, min_len, gap_fill = best[1]
raw = (y_pred_val_nights >= t).astype(int)
ypp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                for i in range(raw.shape[0])])

for i in range(7):
    print(i,
          "true_events", len(extract_events_from_binary_mask(y_valN[i])),
          "pred_events", len(extract_events_from_binary_mask(ypp[i])),
          "true%", y_valN[i].mean()*100,
          "pred%", ypp[i].mean()*100)
    

    print(f"night {i}: true_events={n_true:3d} | pred_events={n_pred:3d} | true%={y_valN[i].mean()*100:5.2f}% | pred%={yb[i].mean()*100:5.2f}%")


0 true_events 20 pred_events 1 true% 1.3888888888888888 pred% 0.06666666666666667
night 0: true_events=154 | pred_events= 83 | true%= 1.39% | pred%= 1.18%
1 true_events 96 pred_events 1 true% 9.56111111111111 pred% 0.044444444444444446
night 1: true_events=154 | pred_events= 83 | true%= 9.56% | pred%= 0.61%
2 true_events 192 pred_events 402 true% 27.23888888888889 pred% 54.05
night 2: true_events=154 | pred_events= 83 | true%=27.24% | pred%=81.56%
3 true_events 21 pred_events 7 true% 1.6277777777777775 pred% 0.43888888888888894
night 3: true_events=154 | pred_events= 83 | true%= 1.63% | pred%= 1.02%
4 true_events 16 pred_events 1 true% 2.9000000000000004 pred% 0.08333333333333334
night 4: true_events=154 | pred_events= 83 | true%= 2.90% | pred%= 0.24%
5 true_events 18 pred_events 1 true% 1.633333333333333 pred% 0.05
night 5: true_events=154 | pred_events= 83 | true%= 1.63% | pred%= 0.12%
6 true_events 154 pred_events 4 true% 13.772222222222222 pred% 0.23888888888888887
night 6: true_ev

In [ ]:
best = (-1, None)

for t in np.linspace(0.40, 0.55, 16):
    raw = (y_pred_val_nights >= t).astype(int)

    for min_len in [3, 5, 8]:
        for gap_fill in [3, 5]:
            ypp = np.stack([
                postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                for i in range(raw.shape[0])
            ])

            f1 = event_f1_nights(ypp, y_valN.astype(int), min_iou=0.3)

            if f1 > best[0]:
                best = (f1, (t, min_len, gap_fill))

print("BEST Event-F1:", best[0])
print("BEST params (t, min_len, gap_fill):", best[1])


BEST Event-F1: 0.15950920245352343
BEST params (t, min_len, gap_fill): (0.55, 8, 3)


In [17]:
X_TEST_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\02_processed\nights_test_norm.h5"

with h5py.File(X_TEST_PATH, "r") as f:
    print(list(f.keys()))


['X_nights', 'subject_ids']


In [20]:

with h5py.File(X_TEST_PATH, "r") as f:
    Xn_test_nights = f["X_nights"][:].astype("float32")
    subj_order_test = f["subject_ids"][:].astype(int)

print("Xn_test_nights:", Xn_test_nights.shape)     # (22, 1800000, 8)
print("subj_order_test:", subj_order_test[:10], "...", subj_order_test[-10:])


Xn_test_nights: (22, 1800000, 8)
subj_order_test: [0 1 2 3 4 5 6 7 8 9] ... [12 13 14 15 16 17 18 19 20 21]


In [21]:
y_pred_test_nights = predict_nights_overlap_mean(
    model, Xn_test_nights, chunk_sec=300, stride_sec=60, fs=100, batch_size=4
)
print("y_pred_test_nights:", y_pred_test_nights.shape)  # (22, 18000)
print("min/max/mean:", y_pred_test_nights.min(), y_pred_test_nights.max(), y_pred_test_nights.mean())


y_pred_test_nights: (22, 18000)
min/max/mean: 0.13873239 0.938192 0.48115352


In [27]:
#y_pred_test_nights to csv file
y_pred_test_nights_df = pd.DataFrame(y_pred_test_nights)
y_pred_test_nights_df.insert(0, 'subject_id', subj_order_test)
y_pred_test_nights_df.to_csv("y_pred_test_nights.csv", index=False)
y_pred_test_nights_df.head(22)

,subject_id,0,1,2,3,4,5,6,7,8,...,17990,17991,17992,17993,17994,17995,17996,17997,17998,17999
0,0,0.496723,0.521852,0.517900,0.490017,0.486453,0.486174,0.488291,0.489657,0.487017,...,0.479626,0.477502,0.476090,0.476441,0.479119,0.481642,0.484100,0.511726,0.520078,0.486630
1,1,0.493870,0.511064,0.508774,0.482680,0.483083,0.483956,0.485265,0.481570,0.480794,...,0.456419,0.456415,0.456463,0.456899,0.458476,0.460178,0.461214,0.491714,0.502536,0.473643
2,2,0.491760,0.508483,0.506988,0.478293,0.479149,0.478418,0.478361,0.477702,0.477663,...,0.466238,0.466195,0.466122,0.466091,0.466000,0.466071,0.466079,0.497829,0.509534,0.480209
3,3,0.502231,0.528229,0.535761,0.525033,0.527842,0.518575,0.523061,0.517185,0.514691,...,0.460448,0.460315,0.460317,0.460311,0.460349,0.460462,0.460673,0.492911,0.505042,0.476214
4,4,0.485259,0.496153,0.490023,0.458842,0.458863,0.458840,0.458814,0.458858,0.458894,...,0.459853,0.459450,0.459371,0.459273,0.459678,0.459629,0.459889,0.492442,0.504313,0.476228
5,5,0.517354,0.556737,0.565059,0.538099,0.532811,0.533179,0.534663,0.536236,0.543877,...,0.509234,0.509086,0.508840,0.508039,0.507656,0.506992,0.506801,0.534725,0.543272,0.507939
6,6,0.496044,0.518068,0.513543,0.486847,0.485785,0.486758,0.486736,0.486251,0.487215,...,0.469584,0.469690,0.469698,0.469738,0.469683,0.469711,0.469718,0.501171,0.512449,0.482734
7,7,0.493550,0.514897,0.512074,0.486857,0.488388,0.488685,0.492984,0.493772,0.492635,...,0.459463,0.459439,0.459439,0.459416,0.459460,0.459447,0.459440,0.491750,0.503905,0.475777
8,8,0.513955,0.565183,0.577343,0.563971,0.565616,0.563341,0.565029,0.571326,0.579666,...,0.507079,0.507138,0.507109,0.507100,0.507051,0.507070,0.507053,0.535021,0.543589,0.508366
9,9,0.520518,0.553838,0.559166,0.545531,0.549563,0.559510,0.557472,0.538694,0.531648,...,0.463780,0.463967,0.463932,0.463856,0.463959,0.463865,0.464096,0.495962,0.507708,0.478715


In [23]:
rows = []

for i, s in enumerate(subj_order_test):
    p = y_pred_test_nights[i]  # (18000,)
    for t, val in enumerate(p):
        rows.append({
            "subject": int(s),
            "t_sec": t,
            "p_apnea": float(val)
        })

night_df = pd.DataFrame(rows)

print(night_df.head())
print(night_df.describe())


   subject  t_sec   p_apnea
0        0      0  0.496723
1        0      1  0.521852
2        0      2  0.517900
3        0      3  0.490017
4        0      4  0.486453
             subject          t_sec        p_apnea
count  396000.000000  396000.000000  396000.000000
mean       10.500000    8999.500000       0.481154
std         6.344297    5196.158975       0.046963
min         0.000000       0.000000       0.138732
25%         5.000000    4499.750000       0.461358
50%        10.500000    8999.500000       0.478949
75%        16.000000   13499.250000       0.505578
max        21.000000   17999.000000       0.938192


In [43]:
from collections import Counter

unique_h5 = np.unique(subj_test)
print("unique subjects in X_test.h5:", unique_h5)
print("n unique:", len(unique_h5))

counts = {int(s): int((subj_test==s).sum()) for s in unique_h5}
print("min/max windows per subject:", min(counts.values()), max(counts.values()))

# se subj_order_test existir:
print("subj_order_test:", subj_order_test, "len:", len(subj_order_test))

missing = [int(s) for s in subj_order_test if s not in set(unique_h5)]
extra   = [int(s) for s in unique_h5 if s not in set(subj_order_test)]
print("missing in X_test.h5 (present in subj_order_test):", missing)
print("extra in X_test.h5 (not in subj_order_test):", extra)


unique subjects in X_test.h5: [22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43]
n unique: 22
min/max windows per subject: 200 200
subj_order_test: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21] len: 22
missing in X_test.h5 (present in subj_order_test): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
extra in X_test.h5 (not in subj_order_test): [22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]


In [76]:

# subjects reais do X_test.h5 (22..43)
subjects_sorted = np.sort(np.unique(subj_test))  # [22..43]
assert len(subjects_sorted) == 22
assert y_pred_test_nights.shape[0] == 22

# sua máscara pós-processada já criada:
# y_test_pp: (22,18000) e y_test_win: (22,200,90)
# se ainda não tiver, recria:
t = 0.535
min_len = 6
gap_fill = 3

raw = (y_pred_test_nights >= t).astype(int)
y_test_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                      for i in range(raw.shape[0])])
y_test_win = y_test_pp.reshape(22, 200, 90)

rows = []
for i, s in enumerate(subjects_sorted):
    idx = np.where(subj_test == s)[0]
    ids_s = ids_test[idx]                 # (200,)
    order = np.argsort(ids_s)
    ids_s = ids_s[order]

    y_s = y_test_win[i][order]            # (200,90) alinhado por ID

    df_s = pd.DataFrame(y_s, columns=[f"y_{k}" for k in range(90)])
    df_s.insert(0, "ID", ids_s)
    rows.append(df_s)

submission_df = pd.concat(rows, ignore_index=True).sort_values("ID").reset_index(drop=True)

mask_cols = [c for c in submission_df.columns if c.startswith("y_")]
print("submission shape:", submission_df.shape)         # (4400, 91)
print("IDs unique:", submission_df["ID"].is_unique)     # True
print("overall %ones:", submission_df[mask_cols].to_numpy().mean() * 100)

assert submission_df.shape[0] == len(ids_test), "Número de linhas não bate com X_test"
assert submission_df["ID"].is_unique, "IDs duplicados na submission"


submission shape: (4400, 91)
IDs unique: True
overall %ones: 8.770454545454545


In [74]:
SUB_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\3lcnn_night\pred_per_night_0.52_12_3.csv"
submission_df.to_csv(SUB_PATH, index=False)
print("✅ Saved:", SUB_PATH)


✅ Saved: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\3lcnn_night\pred_per_night_0.52_12_3.csv


In [72]:

min_len = 12
gap_fill = 3

def apply_postproc_from_probs(y_pred_nights, t, min_len=7, gap_fill=3):
    raw = (y_pred_nights >= t).astype(int)
    y_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                     for i in range(raw.shape[0])])
    return y_pp

for t in [0.50, 0.51, 0.52, 0.53, 0.535, 0.54, 0.55, 0.57]:
    y_pp = apply_postproc_from_probs(y_pred_test_nights, t, min_len=min_len, gap_fill=gap_fill)
    print(f"t={t:.3f} -> %ones={y_pp.mean()*100:.3f}%")


t=0.500 -> %ones=24.212%
t=0.510 -> %ones=13.109%
t=0.520 -> %ones=8.823%
t=0.530 -> %ones=7.141%
t=0.535 -> %ones=6.186%
t=0.540 -> %ones=5.383%
t=0.550 -> %ones=2.220%
t=0.570 -> %ones=0.931%
